# RL4CRN app 13: Habituation hallmarks MMC2

This app evaluates the MMC2-style habituation constraints using hinge penalties, binary-search recovery time, and independent loss-component logging for Comet.

In [ ]:
import os, sys, numpy as np
from pathlib import Path

repo_root = Path.cwd().resolve()
if repo_root.name == "apps":
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

print("Python:", sys.version.split()[0])
print("CWD:", os.getcwd())
print("Repo root:", repo_root)

## 1) Imports

In [ ]:
from RL4CRN.utils.input_interface import Configurator, make_task, print_task_summary
from RL4CRN.utils.default_tasks.HabituationTaskKind import HabituationHallmarksMMC2TaskKind

HabituationHallmarksMMC2TaskKind.pretty_help()

## 2) Template IO/CRN

In [ ]:
from RL4CRN.utils.crn_builders import build_simple_IOCRN

cfg = Configurator.preset("paper")
cfg.solver.algorithm = "CVODE"
cfg.solver.rtol = 1e-3
cfg.solver.atol = 1e-6

species_labels = ["X_1", "X_2", "X_3", "X_4"]
crn, species_labels = build_simple_IOCRN(
    species=species_labels,
    production_input_map={"X_1": "u_1"},
    degradation_input_map={},
    dilution_map={"X_1": 0.1, "X_2": 0.1, "X_3": 0.1, "X_4": 0.1},
    production_map={"X_2": 0.1, "X_3": 0.1},
    output_species="X_4",
    solver=cfg.solver,
)

print("Template CRN built.")
print("num_inputs:", crn.num_inputs)
print("species:", species_labels)

## 3) Reaction library

In [ ]:
from RL4CRN.utils.library_builders import build_MAK_library

library_components = build_MAK_library(crn, species_labels, order=2)
library, M, K, masks = library_components
print("Library built: M=", M, "K=", K)

## 4) MMC2 task parameters

In [ ]:
# Protocol
t_on = 1.0
periods = [5.0, 10.0, 15.0]
pulse_shapes = [(t_on, P - t_on) for P in periods]
n_repeats_pre = 50
n_repeats_post = 50
# The paper evaluates intensity/potentiation/recovery at the slowest listed period.
reference_pulse_shape_index = len(pulse_shapes) - 1
gap_time = 50.0
n_t = 1000
u_values = [1.0]
intensity_values = [1.0, 0.5, 0.25]

# MMC2 habituation and recovery thresholds
eps_h = 0.01
eps_subliminal = 0.005
recovery_tol = 0.05
rt_search_depth = 8
rt_max_gap = 200.0
potentiation_gap_fraction = 0.5

# MMC2 validity constants from the whiteboard/note
delta_p_max = 0.5
monotone_drop_tol = 1e-4
i_max = 2
n_post_min = 2
# Maximum late-peak ratio after the maximum peak: 0.2 means min late peak <= 0.2 * max peak.
delta_min = 0.2
delta_t = 0.05
trough_thr = 0.6
trough_tail = 0.02
trough_count_thr = 0.10
trough_count_max = 5
decrement_weight = 25.0
monotone_weight = 10.0
hard_shape_validity = True

# Loss weights
validity_weight = 1.0
habituation_weight = 1.0
recovery_weight = 1.0
potentiation_weight = 1.0
frequency_weight = 1.0
intensity_weight = 1.0
subliminal_weight = 1.0
ordering_margin = 0.0

In [ ]:
task = make_task(
    template_crn=crn,
    library_components=library_components,
    kind="habituation_hallmarks_mmc2",
    species_labels=species_labels,
    params={
        "pulse_shapes": pulse_shapes,
        "reference_pulse_shape_index": reference_pulse_shape_index,
        "n_repeats_pre": n_repeats_pre,
        "n_repeats_post": n_repeats_post,
        "gap_time": gap_time,
        "n_t": n_t,
        "ic": "from_ss",
        "max_peak": 10.0,
        "min_peak": 0.1,
        "u_values": u_values,
        "intensity_values": intensity_values,
        "eps_h": eps_h,
        "eps_subliminal": eps_subliminal,
        "recovery_tol": recovery_tol,
        "rt_search_depth": rt_search_depth,
        "rt_max_gap": rt_max_gap,
        "potentiation_gap_fraction": potentiation_gap_fraction,
        "delta_p_max": delta_p_max,
        "monotone_drop_tol": monotone_drop_tol,
        "i_max": i_max,
        "n_post_min": n_post_min,
        "delta_min": delta_min,
        "delta_t": delta_t,
        "trough_thr": trough_thr,
        "trough_tail": trough_tail,
        "trough_count_thr": trough_count_thr,
        "trough_count_max": trough_count_max,
        "decrement_weight": decrement_weight,
        "monotone_weight": monotone_weight,
        "hard_shape_validity": hard_shape_validity,
        "validity_weight": validity_weight,
        "habituation_weight": habituation_weight,
        "recovery_weight": recovery_weight,
        "potentiation_weight": potentiation_weight,
        "frequency_weight": frequency_weight,
        "intensity_weight": intensity_weight,
        "subliminal_weight": subliminal_weight,
        "ordering_margin": ordering_margin,
    },
)

print_task_summary(task)
assert len(task.u_list[0]) == crn.num_inputs

## 5) Quick smoke evaluation

In [ ]:
loss, info = task.compute_reward(crn)
hallmark_info = info.get("hallmark_info", {})
print("template loss:", loss)
print("rt:", hallmark_info.get("rt"), "rt strict:", hallmark_info.get("rt_strict"))
print("components:")
for k, v in crn.last_task_info["component_losses"].items():
    print(f"  Component: {k}: {v:.4g}")
print("stored hallmark runs:", len(crn.last_task_info["hallmark_runs"]))

In [ ]:
mmc2_plot_cfg = {
    "figsize": (6.8, 18),
    "alpha": 0.95,
    "xlabel": "time",
    "ylabel": "normalized response",
    "show_peak_trough_markers": True,
    "peak_marker_color": "#111111",
    "trough_marker_color": "#4C78A8",
    "peak_trough_marker_size": 14,
}
mmc2_group_plot_cfg = {
    "figsize": (6.8, 3.0),
    "alpha": 0.95,
    "xlabel": "time",
    "ylabel": "normalized response",
    "show_peak_trough_markers": True,
    "peak_marker_color": "#111111",
    "trough_marker_color": "#4C78A8",
    "peak_trough_marker_size": 14,
}
mmc2_diagnostics_plot_cfg = {"figsize": (8.5, 5.5)}
mmc2_input_color = "#4C78A8"
mmc2_response_color = "#B03A2E"

fig, axes = crn.plot_habituation_hallmarks(
    normalize=True,
    input_color=mmc2_input_color,
    trj_color=mmc2_response_color,
    show_memory_species=False,
    plot_cfg=mmc2_plot_cfg,
)

In [ ]:
fig_diag, axes_diag = crn.plot_habituation_hallmark_diagnostics(
    plot_cfg=mmc2_diagnostics_plot_cfg,
)

## 6) Training configuration

In [ ]:
cfg.train.max_added_reactions = 5
cfg.train.epochs = 301
cfg.train.render_every = 5
cfg.train.seed = 0

cfg.render.n_best = 10
cfg.render.disregarded_percentage = 0.9
cfg.render.mode = {
    "style": "logger",
    "task": "habituation_hallmarks_mmc2",
    "format": "figure",
    "figure_prefix": "MMC2 Habituation",
    "topology": True,
    "hof_n_best": 10,
    "normalize": True,
    "input_color": mmc2_input_color,
    "trj_color": mmc2_response_color,
    "show_memory_species": False,
    "plot_cfg": mmc2_plot_cfg,
    "diagnostics": True,
    "diagnostics_plot_cfg": mmc2_diagnostics_plot_cfg,
    "group_plots": ["MMC2 1", "MMC2 2", "MMC2 3", "MMC2 4", "MMC2 5", "MMC2 6"],
    "group_plot_cfg": mmc2_group_plot_cfg,
}

## 7) Comet logger and trainer

In [ ]:
from datetime import datetime
from RL4CRN.utils.input_interface import make_session_and_trainer

logger = None
if os.environ.get("COMET_API_KEY") and os.environ.get("COMET_WORKSPACE"):
    from pytorch_lightning.loggers import CometLogger
    project_name = os.environ.get("COMET_PROJECT_NAME", "Habituation_hallmarks_MMC2")
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    logger = CometLogger(
        api_key=os.environ["COMET_API_KEY"],
        project=project_name,
        workspace=os.environ["COMET_WORKSPACE"],
        name=f"{project_name}_{timestamp}",
    ).experiment
    logger.log_parameters(hallmark_info.get("constants", {}))

trainer = make_session_and_trainer(cfg, task, logger=logger)

## 8) Train and inspect

In [ ]:
checkpoint_path = "habituation_hallmarks_mmc2_chkpt.pkl"
trainer.run(epochs=cfg.train.epochs, checkpoint_path=checkpoint_path)

In [ ]:
best = trainer.inspect_best(plot=True, normalize=True)
if best is not None:
    print("Best loss:", best.last_task_info.get("reward"))
    print("Component losses:", best.last_task_info.get("component_losses"))

In [ ]:
from RL4CRN.utils.visualizations import topology_graph

hof_envs = list(trainer.s.mult_env.hall_of_fame or [])[:10]
print("HoF entries plotted:", len(hof_envs))

for i, env in enumerate(hof_envs):
    crn_hof = env.state
    print(f"HoF {i} loss:", crn_hof.last_task_info.get("reward"))
    fig, _ = crn_hof.plot_habituation_hallmarks(
        normalize=True,
        input_color=mmc2_input_color,
        trj_color=mmc2_response_color,
        show_memory_species=False,
        plot_cfg=mmc2_plot_cfg,
    )
    fig.suptitle(f"HoF {i} MMC2 Habituation Hallmarks")
    fig_diag, _ = crn_hof.plot_habituation_hallmark_diagnostics(
        plot_cfg=mmc2_diagnostics_plot_cfg,
    )
    fig_diag.suptitle(f"HoF {i} MMC2 Loss Diagnostics")
    if logger is not None:
        logger.log_figure(figure_name=f"MMC2 HoF {i} traces", figure=fig)
        logger.log_figure(figure_name=f"MMC2 HoF {i} diagnostics", figure=fig_diag)

if hof_envs:
    fig_hof_div = topology_graph([env.state for env in hof_envs], t=5, figsize=(10, 10))
    fig_hof_div.suptitle("MMC2 HoF Top-10 Diversity Graph")
    if logger is not None:
        logger.log_figure(figure_name="MMC2 HoF topology diversity", figure=fig_hof_div)

current_top_envs = sorted(
    trainer.s.mult_env.envs,
    key=lambda env: env.state.last_task_info.get("reward", float("inf")),
)[:10]
if current_top_envs:
    fig_batch_div = topology_graph([env.state for env in current_top_envs], t=5, figsize=(10, 10))
    fig_batch_div.suptitle("MMC2 Current Batch Top-10 Diversity Graph")
    if logger is not None:
        logger.log_figure(figure_name="MMC2 current batch topology diversity", figure=fig_batch_div)

In [ ]:
trainer.save(checkpoint_path)